In [0]:
# Read GCP credentials from Databricks secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Configure Spark to access GCS
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

# Test access to the reviews dataset
reviews_path = "gs://amazon_appliance/raw/Appliances.jsonl"
raw_reviews_rdd = sc.textFile(reviews_path)
print("GCP access successful:", bool(raw_reviews_rdd.take(1)))

In [0]:
import json

reviews_rdd = raw_reviews_rdd.map(json.loads)
print("total reviews:", raw_reviews_rdd.count())
print("fields:", list(reviews_rdd.first().keys()))

In [0]:
# check missing values in each field

fields = reviews_rdd.first().keys()
for field in fields:
    missing = reviews_rdd.filter(lambda x: x.get(field) is None or x.get(field) == "").count()
    print(field, ":", missing)

In [0]:
# check invalid ratings
invalid_ratings = reviews_rdd.filter(lambda x: x.get('rating') not in [1, 2, 3, 4, 5]).count()

# check missing join keys
invalid_keys = reviews_rdd.filter(lambda x: not x.get("parent_asin")).count()

print("invalid ratings:", invalid_ratings)
print("invalid parent_asin:", invalid_keys)

In [0]:
# check duplicate reviews

total = raw_reviews_rdd.count()
unique = raw_reviews_rdd.distinct().count()

print("total reviews:", total)
print("unique reviews:", unique)
print("duplicate reviews:", total - unique)

In [0]:
# remove exact duplicate reviews
unique_reviews_rdd = raw_reviews_rdd.distinct().map(json.loads)

# clean and retain required fields
def clean_review(x):
    return {
        "rating": x["rating"],
        "title": x["title"],
        "text": x.get("text") or "",
        "asin": x["asin"],
        "parent_asin": x["parent_asin"],
        "timestamp": x["timestamp"],
        "helpful_vote": x["helpful_vote"],
        "verified_purchase": x["verified_purchase"]
    }

cleaned_reviews_rdd = unique_reviews_rdd.map(clean_review)
print("cleaned review count:", cleaned_reviews_rdd.count())
print("remaining fields:", list(cleaned_reviews_rdd.first().keys()))

In [0]:
import re

# took help from AI to make right regex
def remove_pii(x):
    x = re.sub(r'\S+@\S+\.\S+', '[EMAIL]', x)
    x = re.sub(r'\b\d{3}[-. ]?\d{3}[-. ]?\d{4}\b', '[PHONE]', x)
    return x

def clean_review(x):
    return {
        "rating": x["rating"],
        "title": remove_pii(x["title"]),
        "text": remove_pii(x["text"] or ""),
        "asin": x["asin"],
        "parent_asin": x["parent_asin"],
        "timestamp": x["timestamp"],
        "helpful_vote": x["helpful_vote"],
        "verified_purchase": x["verified_purchase"]
    }

cleaned_reviews_rdd = unique_reviews_rdd.map(clean_review)

print("cleaned reviews:", cleaned_reviews_rdd.count())
print("invalid ratings:", cleaned_reviews_rdd.filter(lambda x: x["rating"] not in [1, 2, 3, 4, 5]).count())
print("missing join keys:", cleaned_reviews_rdd.filter(lambda x: not x["parent_asin"]).count())

In [0]:
# save cleaned reviews to GCP
output_path = "gs://amazon_appliance/processed/reviews_clean/"

# delete old output so the notebook can be rerun
out = sc._jvm.org.apache.hadoop.fs.Path(output_path)
fs = out.getFileSystem(sc._jsc.hadoopConfiguration())
if fs.exists(out):
    fs.delete(out, True)

cleaned_reviews_rdd.map(lambda x: json.dumps(x)).saveAsTextFile(output_path)
print("cleaned reviews saved successfully")

In [0]:
# read cleaned reviews from GCP
saved_reviews_rdd = sc.textFile(output_path).map(json.loads)
print("saved reviews:", saved_reviews_rdd.count())
print("fields:", list(saved_reviews_rdd.first().keys()))
print("missing join keys:", saved_reviews_rdd.filter(lambda x: not x["parent_asin"]).count())